# Setup

In [1]:
#load basic libraries
import polars as pl  # It is advised to use polars, as the detasets can be quite memory-intensive
import numpy as np
import torch.nn as nn
from torch import optim
import torch
import re
from tqdm import trange
import random
from utils import preprocess



%load_ext autoreload
%autoreload 2
%matplotlib inline

datasets = ["ADAUSDT", "BCHUSDT", "DOGEUSDT", "ETHUSDT", "LTCUSDT", "SOLUSDT", "XRPUSDT"]

In [2]:
dataset = "BTCUSDT"
data_root = "data"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_direct = f"{data_root}/{dataset}"
X_train = pl.read_parquet(f"{data_direct}/X_train.parquet")
y_train = pl.read_parquet(f"{data_direct}/y_train.parquet")

# load the volume to fill info from the text file
with open(f"{data_direct}/vol_to_fill.txt", "r") as file:
    content = file.read().strip()

match = re.search(r"The volume to fill is: ([\d.]+)", content)
if match:
    volume_to_fill  = float(match.group(1))
    print(f"Extracted number: {volume_to_fill}")
else:
    print("No number found in the file.")


Extracted number: 4.0


In [3]:
X_train, y_train, means, stds, X_id_map, y_id_map = preprocess(X_train, y_train, device)

# id_map contains the mapping from index along the second dimension to anonymized_id used in the tensor
# X_train, y_train have shape (seq_len, num_ids, num_features)

# TWAP

In [19]:
# Get ID values
id_values = X_id_map[:, 1].cpu().numpy()

# id column
ids_repeated = np.repeat(id_values, 60)

# time_in_hour column
times_seconds = np.arange(59*60, 59*60 + 60) * 1000
unique_ids = X_train.shape[1]
times_repeated = np.tile(times_seconds, unique_ids)

twap_preds = pl.DataFrame({
    'anonymized_id': ids_repeated,
    'time_in_hour': times_repeated,
    'volume_to_fill': volume_to_fill / 60 # Ensure volume is flat
}).with_columns(
    # This cast turns raw integers into the Duration type that prints as '59m 1s'
    pl.col('time_in_hour').cast(pl.Duration(time_unit='ms'))
)

twap_preds

anonymized_id,time_in_hour,volume_to_fill
u64,duration[ms],f64
10076153343292355,59m,0.066667
10076153343292355,59m 1s,0.066667
10076153343292355,59m 2s,0.066667
10076153343292355,59m 3s,0.066667
10076153343292355,59m 4s,0.066667
…,…,…
18444992991527050644,59m 55s,0.066667
18444992991527050644,59m 56s,0.066667
18444992991527050644,59m 57s,0.066667


In [ ]:
# How do you generate predictions again?
simulate_walk_the_book()
def objective_func():
    return